In [1]:
!nvidia-smi

Fri Jun 19 07:37:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:8D:00.0 Off |                    0 |
| N/A   31C    P0             73W /  700W |       0MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
!pip install huggingface_hub transformers accelerate colorama datasets -q

In [2]:
import os

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [3]:
!pip install colorama

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import colorama

device_type = 'cuda'

big_model_name =  "google/gemma-3-1b-pt"
small_model_name = "google/gemma-3-270m"

torch_dtype = torch.bfloat16

In [5]:
input_text = 'once upon'
gamma = 5
seqlen = 30

In [6]:
tokenizer = AutoTokenizer.from_pretrained(big_model_name)

big_model = AutoModelForCausalLM.from_pretrained(
  big_model_name, torch_dtype=torch_dtype, device_map="auto"
)
small_model = AutoModelForCausalLM.from_pretrained(
  small_model_name, torch_dtype=torch_dtype, device_map="auto"
)

print("Models loaded!")
print(f"Big model device:   {next(big_model.parameters()).device}")
print(f"Small model device: {next(small_model.parameters()).device}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/236 [00:00<?, ?it/s]

Models loaded!
Big model device:   cuda:0
Small model device: cuda:0


In [9]:
def generate_draft_and_verify(big_model, small_model, input_ids, gamma, device):
  draft_tokens = []
  q_probs = []
  current_ids = input_ids.clone()

  print(f"\n--- DRAFT PHASE (small model generates {gamma} tokens) ---")
  with torch.no_grad():
      for i in range(gamma):
          out = small_model(current_ids)
          logits = out.logits[:, -1, :]
          vocab_size = logits.shape[-1]
          special_ids = [id for id in tokenizer.all_special_ids if id < vocab_size]
          logits[:, special_ids] = float('-inf')
          probs = torch.softmax(logits, dim=-1)
          token = torch.multinomial(probs.float(), num_samples=1)
          token_str = tokenizer.decode([token.item()])
          top_prob = probs.max().item()
          print(f"  Draft token {i+1}: '{token_str}' (id={token.item()}, top-prob={top_prob:.3f})")
          q_probs.append(probs.squeeze(0))
          draft_tokens.append(token.item())
          current_ids = torch.cat([current_ids, token], dim=-1)

      q = torch.stack(q_probs, dim=0)

      print(f"\n--- VERIFY PHASE (big model scores all {gamma} draft tokens in 1 pass) ---")
      out_big = big_model(current_ids)
      p_logits = out_big.logits[:, -(gamma + 1):, :]
      vocab_size = p_logits.shape[-1]
      special_ids = [id for id in tokenizer.all_special_ids if id < vocab_size]
      p_logits[:, :, special_ids] = float('-inf')
      p = torch.softmax(p_logits.squeeze(0), dim=-1)

      for i in range(gamma):
          draft_tok = draft_tokens[i]
          p_score = p[i, draft_tok].item()
          q_score = q[i, draft_tok].item()
          ratio = min(1.0, p_score / (q_score + 1e-9))
          big_top = tokenizer.decode([p[i].argmax().item()])
          print(f"  Position {i+1}: draft='{tokenizer.decode([draft_tok])}' | "
                f"p={p_score:.3f} q={q_score:.3f} accept_prob={ratio:.2f} | "
                f"big model preferred='{big_top}'")

  draft_token_tensor = torch.tensor(draft_tokens, device=device)
  return q, p, draft_token_tensor

def speculative_sampling(q, p, draft_tokens, gamma, device):
    accepted = []
    print(f"\n--- ACCEPTANCE / REJECTION ---")
    
    for i in range(gamma):
      token_id = draft_tokens[i].item()
      acceptance_prob = torch.clamp(p[i, token_id] / (q[i, token_id] + 1e-9), max=1.0)
      r = torch.rand(1).item()
      token_str = tokenizer.decode([token_id])
    
      if r <= acceptance_prob: 
          print(f"  Token {i+1} '{token_str}': ACCEPTED (r={r:.2f} <= threshold={acceptance_prob:.2f})")
          accepted.append(token_id)
      else:
          corrected = torch.clamp(p[i] - q[i], min=0.0)
          if corrected.sum() < 1e-9:
              corrected = p[i]
          corrected = corrected / corrected.sum()
          resampled = torch.multinomial(corrected.float(), num_samples=1).item()
          resampled_str = tokenizer.decode([resampled])
          print(f"  Token {i+1} '{token_str}': REJECTED (r={r:.2f} > threshold={acceptance_prob:.2f}) "
                f"→ resampled from corrected dist: '{resampled_str}'")
          accepted.append(resampled)
          return accepted, None
    
    bonus = torch.multinomial(p[gamma].float(), num_samples=1).item()
    print(f"  All {gamma} accepted! Bonus token from big model: '{tokenizer.decode([bonus])}'")
    return accepted, bonus

In [11]:
from colorama import Fore, Style, init as colorama_init
import time
colorama_init()

def speculative_decode(big_model, small_model, tokenizer, input_text,
                     gamma=5, seqlen=30, device='cuda'):
  input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(device)
  generated_ids = input_ids.clone()
  total_accepted = 0 
  total_draft = 0
  output_text = input_text
  t0 = time.time()

  print(f"Input: '{input_text}'")
  print(f"Generating {seqlen} tokens with gamma={gamma}...\n")

  step = 0
  while generated_ids.shape[-1] - input_ids.shape[-1] < seqlen:
      step += 1
      g = min(gamma, seqlen - (generated_ids.shape[-1] - input_ids.shape[-1]))
      print(f"========== STEP {step} ==========")

      q, p, draft_tokens = generate_draft_and_verify(big_model, small_model, generated_ids, g, device)
      accepted, bonus = speculative_sampling(q, p, draft_tokens, g, device)

      for tok in accepted:
          total_accepted += 1
          output_text += Fore.GREEN + tokenizer.decode([tok]) + Style.RESET_ALL
      total_draft += g

      if bonus is not None:
          output_text += Fore.YELLOW + tokenizer.decode([bonus]) + Style.RESET_ALL
          accepted.append(bonus)

      new_ids = torch.tensor([accepted], device=device)
      generated_ids = torch.cat([generated_ids, new_ids], dim=-1)

      accepted_strs = [tokenizer.decode([t]) for t in accepted]
      print(f"\n  >> Tokens added this step: {accepted_strs}")

  walltime = time.time() - t0
  acceptance_rate = total_accepted / max(total_draft, 1)

  print(f"\n{'='*40}")
  print(f"DONE — {seqlen} tokens in {walltime:.2f}s")
  print(f"Acceptance rate: {acceptance_rate:.2%}")
  print(f"\nGenerated text:\n{output_text}")
  return generated_ids, acceptance_rate, walltime


# Run it
generated_ids, ar, wt = speculative_decode(
  big_model, small_model, tokenizer,
  input_text, gamma=gamma, seqlen=seqlen, device=device_type
)

Input: 'once upon'
Generating 30 tokens with gamma=5...

========== STEP 1 ==========

--- DRAFT PHASE (small model generates 5 tokens) ---
  Draft token 1: ' a' (id=496, top-prob=0.980)
  Draft token 2: ' time' (id=990, top-prob=0.973)
  Draft token 3: ',' (id=236764, top-prob=0.365)
  Draft token 4: ' in' (id=528, top-prob=0.188)
  Draft token 5: ' the' (id=506, top-prob=0.400)

--- VERIFY PHASE (big model scores all 5 draft tokens in 1 pass) ---
  Position 1: draft=' a' | p=0.977 q=0.980 accept_prob=1.00 | big model preferred=' a'
  Position 2: draft=' time' | p=0.973 q=0.973 accept_prob=1.00 | big model preferred=' time'
  Position 3: draft=',' | p=0.264 q=0.365 accept_prob=0.72 | big model preferred=','
  Position 4: draft=' in' | p=0.087 q=0.061 accept_prob=1.00 | big model preferred=' there'
  Position 5: draft=' the' | p=0.176 q=0.400 accept_prob=0.44 | big model preferred=' a'

--- ACCEPTANCE / REJECTION ---
  Token 1 ' a': ACCEPTED (r=0.78 <= threshold=1.00)
  Token 2 ' time'